# 03 — Merge and Feature Engineering

This notebook merges WQP and USGS to one site-day table and creates:
- time features
- lag features
- rolling means
- compliance flag

In [ ]:
import sys
from pathlib import Path
import yaml
import pandas as pd

project_root = Path("..").resolve()
sys.path.append(str(project_root / "src"))

from features import add_time_features, add_lag_features, add_rolling_mean_features, add_compliance_flag, filter_usable_sites

In [ ]:
cfg = yaml.safe_load(open(project_root / "config" / "config.sample.yaml"))
wqp_wide = pd.read_csv(project_root / "data" / "interim" / "wqp_wide.csv")
usgs_wide = pd.read_csv(project_root / "data" / "interim" / "usgs_wide.csv")
merged = wqp_wide.merge(usgs_wide, on=["usgs_site_no", "date"], how="left")
merged.head()

In [ ]:
merged = add_time_features(merged, date_col="date")
merged = add_lag_features(merged, "usgs_site_no", ["ph", "turbidity", "dissolved_oxygen"], [1,3,7])
merged = add_rolling_mean_features(merged, "usgs_site_no", ["ph", "turbidity", "dissolved_oxygen"], [7,14])
merged = add_compliance_flag(merged, turbidity_threshold=cfg["features"]["compliance_thresholds"]["turbidity_gt"])
merged = filter_usable_sites(
    merged,
    min_rows_per_site=cfg["quality_filters"]["min_rows_per_site_after_merge"],
    min_nonnull_targets_per_site=cfg["quality_filters"]["min_nonnull_targets_per_site"],
)
print("Merged usable rows:", len(merged))
print("Unique sites:", merged["usgs_site_no"].nunique())
merged.head()

In [ ]:
merged.to_parquet(project_root / "data" / "processed" / "final_dataset.parquet", index=False)
merged.to_csv(project_root / "data" / "processed" / "final_dataset.csv", index=False)